# Fast tomo Procedure in Python 

In [1]:
# imports
import tomobase
import os
import ncempy
import stackview
import numpy as np
import pandas as pd 
import logging
tomobase.logger.setLevel(logging.WARNING)
directory = r'\\ematbyname\emat\TimC\USC'
subdirectory = 'B10'
ser_file = 'ftomo2_1.ser'
csv_file = 'angle_log_B10-3.csv'

imgs = []
for i in range(10000):
    try:
        imgs.append(ncempy.io.ser.fileSER(os.path.join(directory, subdirectory, ser_file)).getDataset(i)[0])
    except:
        break
    

data = np.stack(imgs, axis=0)
#stackview.slice(data)

df = pd.read_csv(os.path.join(directory, subdirectory, csv_file))
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed')]

arr = df.iloc[:, :2].astype(float).to_numpy()
shape = data.shape[0]


t = np.linspace(arr[0, 0], arr[-1, 0], shape)

angles = np.zeros(shape, dtype=float)

angles[0] = arr[0, 1]
angles[-1] = arr[-1, 1]
for i in range(0, arr.shape[0]-1):
    mask = (t >= arr[i, 0]) & (t < arr[i+1, 0])
    angles[mask] = arr[i, 1]
    


sino = tomobase.data.Sinogram(data, angles)


In [5]:
import tomobase
import os
import ncempy
import stackview
import numpy as np
import pandas as pd 
import logging
tomobase.logger.setLevel(logging.WARNING)

import tomobase
from tomobase.data import Sinogram
import stackview

sino = Sinogram.from_file(r'\\ematbyname\emattitan\Tim\20242006\run-aub2nu1000temp-23.h5')
sino2 = Sinogram.from_file(r'\\ematbyname\emattitan\Tim\20242006\run-aub2nu1000temp-35.h5')
sino2.times += sino.times[-1]

sino.insert(sino2.data, sino2.angles, sino2.times)
print(sino.times)
print(sino.angles)
stackview.slice(sino.data)

(89, 1024, 1024) (89,) (89,)
(89, 1024, 1024) (89,) (89,)
(22, 1024, 1024) (22,) (22,)
(22, 1024, 1024) (22,) (22,)
[   52.09235915    80.7626718    145.31038856   199.98723085
   235.2687804    329.18315505   397.42896752   445.40664109
   532.7108361    639.59366078   716.78566968   771.24078137
   815.74289497   861.08098262   955.15940439  1013.08415285
  1057.63122135  1147.29273661  1240.93370167  4441.66998005
  4497.86322972  4584.89776158  4655.1964425   4719.58106235
  4760.85477612  4792.26438118  4856.23005948  4956.94709675
  5027.80620562  5092.3529542   5163.33806194  5266.93391967
  5315.06292621  5353.13062046  5420.98588729  5498.47477364
  5575.32011817  5623.41596515  5678.44040379  5754.255504
  5815.48205933  5860.03587745  5918.03194128  5992.33632742
  6053.72681279  6108.25030093  6172.74427641  6234.50001858
  6286.37801699  6354.05652896  6409.08389693  6457.39458222
  6525.20296748  7099.845326    7135.08199325  7199.2703733
  7266.664665    7347.23456071  7

In [4]:
#sino = tomobase.processes.align_sinogram_center_of_mass(sino)
#sino.data.transpose((0,2,1))
#sino = tomobase.processes.bin(sino)
sino = tomobase.processes.background_subtract_median(sino)
sino = tomobase.processes.align_sinogram_xcorr(sino)
sino = tomobase.processes.normalize(sino)
#sino = tomobase.processes.align_tilt_axis_shift(sino)
print(sino.times)

100%|██████████| 111/111 [00:00<00:00, 1588.26it/s]


[   52.09235915    80.7626718    145.31038856   199.98723085
   235.2687804    329.18315505   397.42896752   445.40664109
   532.7108361    639.59366078   716.78566968   771.24078137
   815.74289497   861.08098262   955.15940439  1013.08415285
  1057.63122135  1147.29273661  1240.93370167  4441.66998005
  4497.86322972  4584.89776158  4655.1964425   4719.58106235
  4760.85477612  4792.26438118  4856.23005948  4956.94709675
  5027.80620562  5092.3529542   5163.33806194  5266.93391967
  5315.06292621  5353.13062046  5420.98588729  5498.47477364
  5575.32011817  5623.41596515  5678.44040379  5754.255504
  5815.48205933  5860.03587745  5918.03194128  5992.33632742
  6053.72681279  6108.25030093  6172.74427641  6234.50001858
  6286.37801699  6354.05652896  6409.08389693  6457.39458222
  6525.20296748  7099.845326    7135.08199325  7199.2703733
  7266.664665    7347.23456071  7398.2414717   7459.72966949
  7530.91696294  7749.50783178  7797.66910017  7907.56172239
  7962.10280147  8026.50887

In [ ]:
# translate x and y 
# import circle shift
from scipy.ndimage import center_of_mass, shift, rotate
translate_x = 0
translate_y = -100
sino.data = shift(sino.data, (0, translate_y, translate_x), mode='wrap')
stackview.slice(sino.data)


In [3]:
stackview.crop(sino.data)

_Cropper(children=(HBox(children=(VBox(children=(VBox(children=(IntRangeSlider(value=(0, 111), description='Z'…

In [4]:
import copy
sino_cropped = copy.deepcopy(sino)
cropx = slice(280, 1000)
cropy = slice(290, 1024)
sino_cropped.data = sino_cropped.data[:, cropx, cropy]

sino_cropped = tomobase.processes.align_sinogram_xcorr(sino_cropped)
stackview.slice(sino_cropped.data)

#sino = tomobase.processes.align_tilt_axis_shift(sino)


100%|██████████| 111/111 [00:00<00:00, 489.99it/s]


In [6]:
sino.remove([10,8,34,82,90])
print(sino.times)

[   52.09235915    80.7626718    145.31038856   199.98723085
   235.2687804    329.18315505   397.42896752   445.40664109
   639.59366078   771.24078137   815.74289497   861.08098262
   955.15940439  1013.08415285  1057.63122135  1147.29273661
  1240.93370167  4441.66998005  4497.86322972  4584.89776158
  4655.1964425   4719.58106235  4760.85477612  4792.26438118
  4856.23005948  4956.94709675  5027.80620562  5092.3529542
  5163.33806194  5266.93391967  5315.06292621  5353.13062046
  5498.47477364  5575.32011817  5623.41596515  5678.44040379
  5754.255504    5815.48205933  5860.03587745  5918.03194128
  5992.33632742  6053.72681279  6108.25030093  6172.74427641
  6234.50001858  6286.37801699  6354.05652896  6409.08389693
  6457.39458222  6525.20296748  7099.845326    7135.08199325
  7199.2703733   7266.664665    7347.23456071  7398.2414717
  7459.72966949  7530.91696294  7749.50783178  7797.66910017
  7907.56172239  7962.10280147  8026.50887479  8074.76488407
  8115.90376825  8164.4078

In [5]:
sino = sino_cropped

In [7]:
sino = tomobase.processes.align_sinogram_xcorr(sino)
sino = tomobase.processes.align_tilt_axis_shift(sino)

100%|██████████| 720/720 [00:28<00:00, 25.46it/s]

100%|██████████| 720/720 [00:28<00:00, 25.34it/s]

100%|██████████| 720/720 [00:27<00:00, 26.38it/s]

100%|██████████| 720/720 [00:27<00:00, 26.42it/s]

100%|██████████| 720/720 [00:28<00:00, 25.48it/s]

100%|██████████| 720/720 [00:27<00:00, 26.50it/s]

100%|██████████| 720/720 [00:27<00:00, 26.26it/s]

100%|██████████| 720/720 [00:27<00:00, 26.37it/s]

100%|██████████| 720/720 [00:27<00:00, 26.27it/s]

100%|██████████| 720/720 [00:27<00:00, 25.98it/s]

100%|██████████| 720/720 [00:27<00:00, 25.80it/s]

100%|██████████| 720/720 [00:27<00:00, 26.20it/s]

100%|██████████| 720/720 [00:28<00:00, 25.54it/s]

100%|██████████| 720/720 [00:28<00:00, 25.71it/s]

100%|██████████| 720/720 [00:27<00:00, 25.79it/s]

100%|██████████| 720/720 [00:27<00:00, 25.91it/s]

100%|██████████| 720/720 [00:28<00:00, 25.58it/s]

100%|██████████| 720/720 [00:28<00:00, 25.30it/s]

100%|██████████| 720/720 [00:28<00:00, 25.42it/s]

100%|██████████| 720/720 [00:27

In [ ]:

import numpy as np
from scipy.signal.windows import hann
from skimage.registration import phase_cross_correlation


def get_bad_images_origin(sino, recon_iters=1, upsample=10, poly_order=2, thresh=1.5):
    s = copy.deepcopy(sino)
    # normalize intensity per-projection (optional)
    s_min = s.data.min()
    s_max = s.data.max()
    s.data = (s.data - s_min) / (s_max)
    # reconstruct and forward-project (use same functions you used before)
    rec = tomobase.processes.optomo_reconstruct(s, iterations=recon_iters)
    sino_sim = tomobase.processes.project(rec, s.angles)
    for i in range(sino_sim.data.shape[0]):
        sino_sim.data[i,:, :] = sino_sim.data[i, :, :] / np.sum(sino_sim.data[i, :, :])
    sino_sim.data = (sino_sim.data - sino_sim.data.min()) / (sino_sim.data.max())
    #sino_sim.data = (sino_sim.data - sino_sim.data.min()) / (sino_sim.data.max() - sino_sim.data.min())
    # optional: equalize intensity per-projection if you have that function
    # sino_sim = hv_EqualizeIntensity(sino_sim)
    n = s.data.shape[0]
    diff = np.zeros_like(s.angles, dtype=float)
    for i in range(n):
        ref = sino_sim.data[i, :, :]
        mov = s.data[i, :, :]
        # compute subpixel shift; returns (shift_y, shift_x), error, phasediff
        shift, error, phasediff = phase_cross_correlation(ref, mov, upsample_factor=upsample)
        diff[i] = float(np.abs(error))
    # detrend and score as in your code
    coeffs = np.polyfit(s.angles, diff, poly_order)
    trend = np.polyval(coeffs, s.angles)
    residuals = diff - trend
    residuals -= residuals.min()
    median = np.median(residuals)
    MAD = np.median(np.abs(residuals - median))
    if MAD == 0:
        MAD = 1e-6
    score = (0.675 * (residuals - median)) / MAD
    idx = np.where(score > thresh)[0]  # zero-based indices
    return sino_sim, idx, score


def _prep_slice(img):
    x = np.asarray(img, dtype=np.float32)
    if np.isnan(x).any():
        x = np.nan_to_num(x, nan=float(np.nanmean(x)))
    m = float(np.mean(x)); s = float(np.std(x))
    x = (x - m) / s if s > 0 else x*0.0
    return x

def _hann2d(h, w):
    wy = hann(h, sym=False).astype(np.float32)
    wx = hann(w, sym=False).astype(np.float32)
    return wy[:, None] * wx[None, :]

def _rolling_median(y, win):
    # 1D rolling median with edge reflection
    y = np.asarray(y, float)
    win = max(3, int(win) | 1)  # odd >=3
    pad = win // 2
    yp = np.pad(y, pad, mode='reflect')
    out = np.empty_like(y)
    for i in range(len(y)):
        out[i] = np.median(yp[i:i+win])
    return out

def get_bad_images(sino, recon_iters=20, upsample=30, poly_order=None, thresh=3,
                   use_mean_std=False, combine_shift=False, overlap_ratio=0.1, crop_center=None):
    """
    Returns:
      idx: indices of likely-bad projections (0-based)
      score: robust z-like score (higher = worse)
      diff: raw metric before baseline removal (higher = worse)
    """
    s = copy.deepcopy(sino)
    s.data = (s.data-np.min(s.data))/(np.max(s.data)-np.min(s.data))
    # Reconstruct & forward-project
    rec = tomobase.processes.optomo_reconstruct(s, iterations=recon_iters)
    sino_sim = tomobase.processes.project(rec, s.angles)

    n, H, W = s.data.shape
    assert sino_sim.data.shape == s.data.shape, \
        f"Shape mismatch: {sino_sim.data.shape} vs {s.data.shape}"

    win2d = _hann2d(H, W)
    diff_error = np.zeros(n, dtype=float)
    diff_shift = np.zeros(n, dtype=float)

    for i in range(n):
        ref = _prep_slice(sino_sim.data[i])
        mov = _prep_slice(s.data[i])

        if crop_center:
            cy, cx = H//2, W//2
            hy, hx = int(H*crop_center/2), int(W*crop_center/2)
            ys, ye = cy-hy, cy+hy
            xs, xe = cx-hx, cx+hx
            ref = ref[ys:ye, xs:xe]
            mov = mov[ys:ye, xs:xe]
            win = _hann2d(ref.shape[0], ref.shape[1])
        else:
            win = win2d

        ref *= win
        mov *= win

        shift, error, _ = phase_cross_correlation(
            ref, mov,
            upsample_factor=upsample,
            overlap_ratio=overlap_ratio
        )
        diff_error[i] = float(error)                     # 0 (good) → 1 (bad)
        diff_shift[i] = float(np.hypot(shift[0], shift[1]))  # pixels, >=0

    # Pick metric: PCC error is intensity-invariant like your MATLAB p(1).
    diff = diff_error.copy()
    if combine_shift:
        # Blend in shift magnitude to help when error saturates
        # normalize to comparable scale
        s_norm = diff_shift / (np.median(diff_shift) + 1e-6)
        diff = 0.7*diff_error + 0.3*np.tanh(0.3*s_norm)

    # ---- Baseline removal (robust) ----
    # Use rolling median instead of high-order poly (prevents overfitting spikes).
    # Window ~5–11 projections often works well; tune if your angles are dense.
    baseline = _rolling_median(diff, win=11)
    residuals = diff - baseline

    # ---- Robust scoring ----
    if use_mean_std:
        mu = float(np.mean(residuals))
        sd = float(np.std(residuals)) or 1e-6
        score = (residuals - mu) / sd
    else:
        med = float(np.median(residuals))
        MAD = float(np.median(np.abs(residuals - med))) or 1e-6
        # 0.675 ~ 1/1.4826 to map MAD to sigma
        score = (0.675 * (residuals - med)) / MAD

    idx = np.where(score > thresh)[0]
    return sino_sim, idx, score


In [ ]:
import numpy as np

# -------------------- frequency helpers --------------------

def _fftfreq_grids(shape, voxel_size=1.0):
    freqs = [np.fft.fftfreq(n, d=voxel_size) for n in shape]
    fx, fy, fz = np.meshgrid(*freqs, indexing="ij")
    r = np.sqrt(fx**2 + fy**2 + fz**2)  # radial spatial frequency (cycles / unit)
    return fx, fy, fz, r

def _odd_even_axis_masks(shape, axis):
    idx = [np.arange(n, dtype=np.int32) for n in shape]
    grids = np.meshgrid(*idx, indexing="ij")
    a = grids[axis] % 2
    even_mask = (a == 0)
    odd_mask  = ~even_mask
    return even_mask, odd_mask

# -------------------- optional preprocessing --------------------

def _apodize_mask(mask, power=3.0):
    """Softens a real-space mask edge by raising it to a power."""
    m = np.clip(mask.astype(np.float32), 0, 1)
    return m ** power

def _apply_mask(vol, mask=None):
    if mask is None:
        return vol
    return vol * _apodize_mask(mask)

# -------------------- low-pass along one axis --------------------

def _lowpass_axis(arr, axis, voxel_size=1.0, cutoff_axis=None, trans=0.0):
    """
    Low-pass filter in Fourier domain, limiting ONLY the chosen axis.
    cutoff_axis (cycles/unit): default is 0.25/voxel_size for decimation-by-2 along that axis.
    trans (cycles/unit): optional raised-cosine transition width (0 for a hard box).
    """
    if cutoff_axis is None:
        cutoff_axis = 0.25 / voxel_size

    F = np.fft.fftn(arr)
    fx, fy, fz, _ = _fftfreq_grids(arr.shape, voxel_size)

    if axis == 0: fa = np.abs(fx)
    elif axis == 1: fa = np.abs(fy)
    else: fa = np.abs(fz)

    if trans > 0:
        inner = fa <= (cutoff_axis - trans)
        outer = fa >= (cutoff_axis + trans)
        mid = (~inner) & (~outer)
        w = np.zeros_like(fa, dtype=np.float32)
        w[inner] = 1.0
        # raised-cosine ramp (width = 2*trans)
        w[mid] = 0.5 * (1.0 + np.cos(np.pi * (fa[mid] - (cutoff_axis - trans)) / (2.0 * trans)))
        F *= w
    else:
        F *= (fa <= cutoff_axis)

    return np.fft.ifftn(F).real

# -------------------- FSC core --------------------

def _fsc_between(vol1, vol2, voxel_size=1.0, edges=None, n_shells=None, eps=1e-12):
    if vol1.shape != vol2.shape:
        raise ValueError("vol1 and vol2 must have the same shape")

    v1 = vol1.astype(np.float32) - np.mean(vol1)
    v2 = vol2.astype(np.float32) - np.mean(vol2)

    F1 = np.fft.fftn(v1)
    F2 = np.fft.fftn(v2)

    _, _, _, r = _fftfreq_grids(v1.shape, voxel_size)
    if edges is None:
        r_max = r.max()
        if n_shells is None:
            n_shells = min(v1.shape) // 2
        edges = np.linspace(0.0, r_max, n_shells + 1)

    bins = np.digitize(r.ravel(), edges) - 1
    n = edges.size - 1

    cross = (F1 * np.conj(F2)).ravel().real
    p1 = (np.abs(F1)**2).ravel()
    p2 = (np.abs(F2)**2).ravel()

    valid = (bins >= 0) & (bins < n)
    b = bins[valid]
    num  = np.bincount(b, weights=cross[valid], minlength=n)
    den1 = np.bincount(b, weights=p1[valid],    minlength=n)
    den2 = np.bincount(b, weights=p2[valid],    minlength=n)

    fsc = num / (np.sqrt(den1 * den2) + eps)
    freqs = 0.5 * (edges[:-1] + edges[1:])
    return freqs, fsc

# -------------------- Paper-faithful SFSC --------------------

def sfsc_cardinal(volume, voxel_size=1.0, df=0.01, mask=None,
                  limit_to_decimated_nyquist=True, lpf_transition=0.0):
    """
    Self-FSC per the 2024 paper:
      For each axis a in {x,y,z}:
        1) (optional) apply real-space mask with apodization.
        2) Anti-alias low-pass along axis a to 0.25 / voxel_size.
        3) Split into even/odd slices along axis a.
        4) Reconstruct each sparse half by the same axis low-pass (zero-insert + LPF).
        5) Compute FSC_a between the reconstructions.
      Return SFSC = average(FSC_x, FSC_y, FSC_z).

    Parameters
    ----------
    df : float     Frequency step for bin edges (cycles / unit).
    mask : array   Optional real-space mask (same shape as volume).
    limit_to_decimated_nyquist : bool  If True, cap x-axis at 0.25/voxel_size (recommended).
    lpf_transition : float  Raised-cosine transition width (cycles/unit), e.g. 0.01 for gentler edges.
    """
    vol = _apply_mask(volume.astype(np.float32), mask)

    fN_full = 0.5 / voxel_size
    f_cap   = (0.25 / voxel_size) if limit_to_decimated_nyquist else fN_full
    edges   = np.arange(0.0, f_cap + 1e-12, df)

    fsc_curves = []
    for axis in (0, 1, 2):
        # 1) Anti-alias along this axis
        pre = _lowpass_axis(vol, axis=axis, voxel_size=voxel_size,
                            cutoff_axis=0.25/voxel_size, trans=lpf_transition)

        # 2) Odd/even split along this axis
        even_mask, odd_mask = _odd_even_axis_masks(pre.shape, axis)
        even_sparse = np.zeros_like(pre); even_sparse[even_mask] = pre[even_mask]
        odd_sparse  = np.zeros_like(pre);  odd_sparse[odd_mask]  = pre[odd_mask]

        # 3) Reconstruct both halves to the full grid via same LPF
        rec_even = _lowpass_axis(even_sparse, axis=axis, voxel_size=voxel_size,
                                 cutoff_axis=0.25/voxel_size, trans=lpf_transition)
        rec_odd  = _lowpass_axis(odd_sparse,  axis=axis, voxel_size=voxel_size,
                                 cutoff_axis=0.25/voxel_size, trans=lpf_transition)

        # 4) FSC for this axis
        freqs, fsc_a = _fsc_between(rec_even, rec_odd, voxel_size=voxel_size, edges=edges)
        fsc_curves.append(fsc_a)

    sfsc = np.mean(np.vstack(fsc_curves), axis=0)
    return freqs, sfsc, tuple(fsc_curves)  # order: (FSC_x, FSC_y, FSC_z)

# -------------------- utilities --------------------

def fsc_resolution(freqs, fsc, threshold=0.143):
    """
    Linear interpolation to find resolution (1/f_cut) at first crossing of 'threshold'.
    Returns None if the curve never drops below threshold.
    """
    below = np.where(fsc < threshold)[0]
    if below.size == 0:
        return None
    i = below[0]
    if i == 0:
        f_cut = freqs[0]
    else:
        f1, f2 = freqs[i-1], freqs[i]
        y1, y2 = fsc[i-1], fsc[i]
        if y2 == y1:
            f_cut = f2
        else:
            f_cut = f1 + (threshold - y1) * (f2 - f1) / (y2 - y1)
    return 1.0 / f_cut

In [ ]:
import numpy as np

def projections_to_remove_per_angle(angles, thresh, angle_tol=1e-6):
    """
    Given:
      angles : 1D array of length N (may contain repeated angles)
      score  : 1D array of length N, higher = worse (from get_bad_images)
      thresh : float, outlier threshold on 'score'
      angle_tol : float, tolerance for grouping angles that should be equal

    Returns:
      remove_idx : 1D array of indices to remove.

    Properties:
      - At least ONE projection per (unique) angle is ALWAYS kept.
      - If any projection at an angle has score <= thresh, one such
        "inlier" with the LOWEST score is kept.
      - If ALL projections at an angle are > thresh, the least-bad
        one is kept and the rest are marked for removal.
    """
    angles = np.asarray(angles)
    score  = np.asarray(score)
    N = angles.size
    assert score.shape == (N,)

    # Group "same" angles together using a tolerance
    keys = np.round(angles / angle_tol).astype(np.int64)
    uniq_keys, inv = np.unique(keys, return_inverse=True)

    keep = np.zeros(N, dtype=bool)

    for g in range(len(uniq_keys)):
        group = np.where(inv == g)[0]   # indices for this angle
        if group.size == 0:
            continue

        s_group = score[group]
        inlier = s_group <= thresh

        if np.any(inlier):
            # Keep the best inlier (lowest score)
            candidates = group[inlier]
            best = candidates[np.argmin(score[candidates])]
        else:
            # All projections at this angle are outliers:
            # still keep the least-bad one.
            best = group[np.argmin(s_group)]

        keep[best] = True

    remove_idx = np.where(~keep)[0]
    return remove_idx

In [ ]:
import numpy as np

# ---------- helper: per-projection "resolution"/quality metric ----------

def _projection_quality(img, voxel_size=1.0):
    """
    Per-projection 'resolution' proxy.
    Higher value ≈ more high-frequency content ≈ sharper image.

    Works best when comparing projections with the same geometry & exposure.
    """
    x = np.asarray(img, dtype=np.float32)
    x = x - np.mean(x)

    F = np.fft.fft2(x)
    P = np.abs(F)**2  # power spectrum

    H, W = x.shape
    fy = np.fft.fftfreq(H, d=voxel_size)
    fx = np.fft.fftfreq(W, d=voxel_size)
    FX, FY = np.meshgrid(fx, fy, indexing="xy")
    r = np.sqrt(FX**2 + FY**2)

    r_max = np.max(r)
    if r_max == 0:
        return 0.0

    # weight high frequencies more strongly
    w = (r / r_max)**2
    num = np.sum(P * w)
    den = np.sum(P) + 1e-12
    return float(num / den)


# ---------- main helper: dual best-per-angle + removal list ----------

def plan_projection_removal_per_angle(
    sino,
    recon_iters=20,
    upsample=30,
    angle_tol=1e-6,
    voxel_size=1.0,
    **get_bad_images_kwargs,
):
    """
    For a sinogram that may have multiple projections per angle, determine:
      - the best projection per angle by *resolution/quality*,
      - the best projection per angle by *least outlier* (smallest score),
      - a removal list containing *all other* projections.

    This function:
      1) Runs get_bad_images(...) to obtain a robust outlier score for each
         projection (no thresholding used here).
      2) Computes a per-projection quality metric (higher ≈ higher resolution).
      3) Groups projections by angle (within 'angle_tol').
      4) For each angle group:
           - best_by_score   = argmin(score[group])
           - best_by_quality = argmax(quality[group])
         Both are marked to KEEP (they may be the same index).
      5) Returns:
           - keep_idx   : union of all best_by_score and best_by_quality,
                         sorted by angle.
           - remove_idx : all remaining indices.
           - info       : diagnostics including per-angle choices.

    Important:
      - The function does NOT modify 'sino'.
      - It does NOT apply any hard outlier threshold; "least outlier"
        is purely "smallest score from get_bad_images".

    Parameters
    ----------
    sino : object
        Sinogram-like object with:
          - sino.angles : 1D array of length N
          - sino.data   : array of shape (N, H, W)
    recon_iters : int
        Passed to get_bad_images (reconstruction iterations).
    upsample : int
        Passed to get_bad_images (phase_cross_correlation upsample factor).
    angle_tol : float
        Tolerance for grouping angles that "should be equal".
        Two angles belong to the same group if:
            round(angle / angle_tol) is the same integer.
    voxel_size : float
        Physical pixel size (for frequency scaling in the quality metric).
    **get_bad_images_kwargs :
        Extra kwargs forwarded directly to get_bad_images, e.g.:
          - poly_order
          - thresh
          - use_mean_std
          - combine_shift
          - overlap_ratio
          - crop_center

    Returns
    -------
    keep_idx : np.ndarray (1D, int)
        Indices of projections to KEEP (size = up to 2 * n_angles, but
        smaller if best-by-score == best-by-quality for some angles).
    remove_idx : np.ndarray (1D, int)
        Indices of projections to REMOVE.
    info : dict
        Diagnostics:
          - 'score'           : 1D array, score from get_bad_images (length N)
          - 'quality'         : 1D array, quality metric (length N)
          - 'sino_sim'        : forward-projected sinogram from get_bad_images
          - 'best_by_score'   : 1D array of length n_angle_groups,
                                index of least-outlier per angle
          - 'best_by_quality' : 1D array of length n_angle_groups,
                                index of highest-quality per angle
          - 'angle_rep'       : representative angle per group (same length as above)
    """
    # --- 1) outlier scoring via your existing function ---
    # We ignore the idx output here; we only need the score array.
    sino_sim, _, score = get_bad_images(
        sino,
        recon_iters=recon_iters,
        upsample=upsample,
        **get_bad_images_kwargs,
    )

    angles = np.asarray(sino.angles)
    N = angles.size
    score = np.asarray(score, dtype=float)
    assert score.shape == (N,), "score must be 1D with length N"

    # --- 2) per-projection quality metric (resolution-ish) ---
    quality = np.zeros(N, dtype=float)
    for i in range(N):
        quality[i] = _projection_quality(sino.data[i], voxel_size=voxel_size)

    # --- 3) group by angle and pick best-by-score & best-by-quality ---
    keys = np.round(angles / angle_tol).astype(np.int64)
    uniq_keys, inv = np.unique(keys, return_inverse=True)

    keep = np.zeros(N, dtype=bool)
    best_by_score = []
    best_by_quality = []
    angle_rep = []

    for g, key in enumerate(uniq_keys):
        group = np.where(inv == g)[0]
        if group.size == 0:
            continue

        s_group = score[group]
        q_group = quality[group]

        # least outlier = smallest score
        idx_score_local = np.argmin(s_group)
        best_score_idx = group[idx_score_local]

        # best resolution = highest quality
        idx_quality_local = np.argmax(q_group)
        best_quality_idx = group[idx_quality_local]

        keep[best_score_idx] = True
        keep[best_quality_idx] = True

        best_by_score.append(best_score_idx)
        best_by_quality.append(best_quality_idx)
        # representative angle for this group (e.g. mean or first)
        angle_rep.append(np.mean(angles[group]))

    best_by_score = np.array(best_by_score, dtype=int)
    best_by_quality = np.array(best_by_quality, dtype=int)
    angle_rep = np.array(angle_rep, dtype=float)

    # --- 4) assemble keep/remove indices ---
    keep_idx = np.where(keep)[0]
    remove_idx = np.where(~keep)[0]

    # sort keep_idx by angle for nice ordering
    order = np.argsort(angles[keep_idx])
    keep_idx = keep_idx[order]

    info = {
        "score": score,
        "quality": quality,
        "sino_sim": sino_sim,
        "best_by_score": best_by_score,
        "best_by_quality": best_by_quality,
        "angle_rep": angle_rep,
    }

    return keep_idx, remove_idx, info


In [ ]:
# rotate sinogram 90 degrees
sino.data = np.transpose(sino.data, (0,2,1))
stackview.slice(sino.data)

In [ ]:
import copy
#sino = tomobase.processes.align_tilt_axis_shift(sino)
#sino = tomobase.processes.align_tilt_axis_rotation(sino)
# sino_test, idx, score = get_bad_images_origin(sino, recon_iters=1, poly_order=12)
idx, removal, info = plan_projection_removal_per_angle(sino)
print(info)


stackview.curtain(info["sino_sim"].data, sino.data)
#stackview.curtain(info["sino_sim"].data, sino.data)
#stackview.slice( sino.data)

In [ ]:
table = pd.DataFrame({
    "angle": sino.angles,
    "score": info["score"],
    "quality": info["quality"],
    "best_score": [("YES" if i in info["best_by_score"] else "NO") for i in range(len(sino.angles))],
    "best_quality": [("YES" if i in info["best_by_quality"] else "NO") for i in range(len(sino.angles))],
})
pd.set_option('display.max_rows', None)
print(table)

In [12]:
sino = new_sino


In [13]:
import copy 

new_sino = copy.deepcopy(sino)
new_sino = tomobase.processes.background_subtract_median(sino, inplace=False)
new_sino, offsets_shift = tomobase.processes.align_tilt_axis_shift(new_sino, inplace=False, verbose_outputs=True)
new_sino, offset_angle = tomobase.processes.align_tilt_axis_rotation(new_sino, inplace=False, verbose_outputs=True)

print(np.mean(offsets_shift), offset_angle)
stackview.side_by_side(sino.data, new_sino.data)

100%|██████████| 734/734 [00:28<00:00, 26.16it/s]

100%|██████████| 734/734 [00:27<00:00, 26.37it/s]

100%|██████████| 734/734 [00:27<00:00, 26.45it/s]

100%|██████████| 734/734 [00:29<00:00, 25.22it/s]

100%|██████████| 734/734 [00:29<00:00, 24.68it/s]

100%|██████████| 734/734 [00:28<00:00, 25.39it/s]

100%|██████████| 734/734 [00:29<00:00, 24.57it/s]

100%|██████████| 734/734 [00:28<00:00, 25.42it/s]

100%|██████████| 734/734 [00:28<00:00, 25.80it/s]

100%|██████████| 734/734 [00:28<00:00, 25.57it/s]

100%|██████████| 734/734 [00:28<00:00, 25.47it/s]

100%|██████████| 734/734 [00:28<00:00, 25.58it/s]

100%|██████████| 734/734 [00:27<00:00, 26.45it/s]

100%|██████████| 734/734 [00:27<00:00, 26.37it/s]

100%|██████████| 734/734 [00:27<00:00, 26.48it/s]

100%|██████████| 734/734 [00:27<00:00, 26.25it/s]

100%|██████████| 734/734 [00:27<00:00, 26.23it/s]

100%|██████████| 734/734 [00:27<00:00, 26.47it/s]

100%|██████████| 734/734 [00:28<00:00, 25.93it/s]

100%|██████████| 734/734 [00:28

-10.0 4
side_by_side


In [ ]:
from scipy.ndimage import center_of_mass, shift, rotate
sino.data = shift(sino.data, (0, 100, 0), mode='wrap')

stackview.slice(sino.data)

In [2]:
from tomobase.data import Sinogram
sino = Sinogram.from_file(os.path.join(directory, subdirectory, 'sino_aligned.mrc'))

In [ ]:
import stackview
#compare stack with next image
stackview.side_by_side(sino.data[1:], sino.data[:-1])

In [ ]:


idx =[51,52]



In [ ]:
sino.remove(idx)

sino = tomobase.processes.align_sinogram_xcorr(sino)
#sino = tomobase.processes.align_tilt_axis_shift(sino)

In [10]:
import stackview
sino.data = sino.data.transpose((0,2,1))
stackview.slice(sino.data)

In [ ]:
#stackview.slice(sino.data)

#new_sino=copy.deepcopy(sino)
new_sino = tomobase.processes.align_tilt_axis_shift(new_sino)
#stackview.slice(new_sino.data)

In [ ]:

new_sino = tomobase.processes.align_tilt_axis_rotation(new_sino)
stackview.slice(new_sino.data)

In [ ]:

rec = tomobase.processes.reconstruct.optomo_reconstruct(new_sino, iterations=150)
rec.to_file(os.path.join(directory, subdirectory, 'result_rotfixed.rec'))
stackview.orthogonal(rec.data)

100%|██████████| 247/247 [02:30<00:00,  1.64it/s]


In [7]:
import os
new_sino = Sinogram.from_file(os.path.join(r'C:\Users\TCraig\Pictures\Rod\DIPs', 'mof.mrc'))
rec = reconstruct_tvm(new_sino)
rec.to_file(os.path.join(r'C:\Users\TCraig\Pictures\Rod\DIPs', 'result_rotfixed.rec'))
stackview.orthogonal(rec.data)

MemoryError: Unable to allocate 1.44 GiB for an array with shape (732, 734, 720) and data type float32

In [6]:
import tomosipo as ts
import numpy as np
from tomobase.data import Volume, Sinogram, volume
import cupy as cp

def project(volume, angles):
    """
    Project a 3D volume to generate sinograms at specified angles.

    Parameters:
    - volume: 3D numpy array of shape (num_slices, height, width)
    - angles: 1D numpy array of projection angles in radians

    Returns:
    - sinograms: 3D numpy array of shape (num_angles, num_detectors, num_slices)
    """
    volume.data = volume.data.transpose(2,0,1)  # (S,H,W)->(W,S,H) 021
    volume.data = cp.asarray(volume.data)

    angles_rad = np.radians(angles+90)
    vg = ts.volume(shape=(128,128,128))
    pg = ts.parallel(angles=angles_rad, shape=(128, 128))
    A = ts.operator(vg, pg)


    sinogram = cp.asnumpy(A(volume.data).transpose(1,0, 2))
    return Sinogram(sinogram, angles)

def reconstruct_tvm(sinogram, vol_file=None, num_iterations=100, lambda_tv=0.1 ):
    """
    Reconstruct a 3D volume using SIRT with Total Variation regularization.
    
    Parameters:
    - sinogram: Sinogram object with projection data
    - num_iterations: Number of iterations
    - lambda_tv: TV regularization strength (higher = more smoothing)
    
    Returns:
    - volume: Reconstructed Volume object
    """
    angles_rad = np.radians(sinogram.angles + 90)
    vg = ts.volume(shape=(sinogram.data.shape[1],sinogram.data.shape[1], sinogram.data.shape[2]), size=(1, 1, 1))
    pg = ts.parallel(angles=angles_rad, shape=(sinogram.data.shape[1], sinogram.data.shape[2]), size=(1.0, 1.0))
    
    A = ts.operator(vg, pg)
    
    # SIRT weights
    R =  1 / A(np.ones(A.domain_shape))
    R = np.clip(R, a_min=None, a_max=1 / ts.epsilon)

    C = 1 / A.T(np.ones(A.range_shape))
    C = np.clip(C, a_min=None, a_max=1 / ts.epsilon)

    
    y = sinogram.data.transpose(1, 0, 2)
    if vol_file is None:
        x_rec = np.zeros(A.domain_shape, dtype=np.float32)
    else:
        x_rec = Volume.from_file(vol_file).data
        num_iterations =  int(vol_file.name.split('_')[0]) 
    
    for i in range(num_iterations):
        # Standard SIRT update
        residual = y - A(x_rec)
        sirt_update = C * A.T(R * residual)
        
        # TV gradient (using finite differences)
        tv_grad = compute_tv_gradient(x_rec)
        
        # Combined update with TV regularization
        x_rec += sirt_update - lambda_tv * tv_grad
        #x_rec += sirt_update
        
        # Non-negativity constraint (optional but common in tomography)
        x_rec = np.maximum(x_rec, 0)
    

    x_rec = x_rec.transpose(1, 2, 0)  # (W,S,H)->(S,H,W) 120
    return Volume(x_rec)


def compute_tv_gradient(volume):
    """
    Compute the gradient of the Total Variation functional.
    Uses isotropic TV: sum of gradient magnitudes.
    """
    # Compute gradients along each axis using finite differences
    grad_x = np.zeros_like(volume)
    grad_y = np.zeros_like(volume)
    grad_z = np.zeros_like(volume)
    
    # Forward differences
    grad_x[:-1, :, :] = volume[1:, :, :] - volume[:-1, :, :]
    grad_y[:, :-1, :] = volume[:, 1:, :] - volume[:, :-1, :]
    grad_z[:, :, :-1] = volume[:, :, 1:] - volume[:, :, :-1]
    
    # Gradient magnitude (with small epsilon for numerical stability)
    eps = 1e-8
    grad_mag = np.sqrt(grad_x**2 + grad_y**2 + grad_z**2 + eps)
    
    # Compute TV gradient (divergence of normalized gradient)
    tv_grad = np.zeros_like(volume)
    
    # Backward divergence for x
    div_x = np.zeros_like(volume)
    div_x[1:-1, :, :] = (grad_x[1:-1, :, :] / grad_mag[1:-1, :, :] - 
                          grad_x[:-2, :, :] / grad_mag[:-2, :, :])
    div_x[0, :, :] = grad_x[0, :, :] / grad_mag[0, :, :]
    div_x[-1, :, :] = -grad_x[-2, :, :] / grad_mag[-2, :, :]
    
    # Backward divergence for y
    div_y = np.zeros_like(volume)
    div_y[:, 1:-1, :] = (grad_y[:, 1:-1, :] / grad_mag[:, 1:-1, :] - 
                          grad_y[:, :-2, :] / grad_mag[:, :-2, :])
    div_y[:, 0, :] = grad_y[:, 0, :] / grad_mag[:, 0, :]
    div_y[:, -1, :] = -grad_y[:, -2, :] / grad_mag[:, -2, :]
    
    # Backward divergence for z
    div_z = np.zeros_like(volume)
    div_z[:, :, 1:-1] = (grad_z[:, :, 1:-1] / grad_mag[:, :, 1:-1] - 
                          grad_z[:, :, :-2] / grad_mag[:, :, :-2])
    div_z[:, :, 0] = grad_z[:, :, 0] / grad_mag[:, :, 0]
    div_z[:, :, -1] = -grad_z[:, :, -2] / grad_mag[:, :, -2]
    
    tv_grad = -(div_x + div_y + div_z)
    
    return tv_grad

In [ ]:
#prepare mask for inpainting
rec_masked = tomobase.processes.reconstruct.optomo_reconstruct(new_sino, iterations=50)
thresh = 5

# create cylindrical mask
H, W = rec_masked.data.shape[1], rec_masked.data.shape[2]
radius = int(round(H / 2.0)) - thresh
cy = (H - 1) / 2.0
cx = (W - 1) / 2.0
y, x = np.ogrid[:H, :W]
disk = ((y - cy) ** 2 + (x - cx) ** 2) <= (radius ** 2)   # boolean mask (H,W)
cylindrical_mask = disk[None, :, :].astype(rec_masked.data.dtype)     # (H,W,1) -> broadcast (H,W,Z)
rec_masked.data *= cylindrical_mask
stackview.orthogonal(rec_masked.data)

In [ ]:
# Python conversion of:
# threshold = 0.08;
# n_pixels_to_grow = 1;
# mask = imdilate(reproj > threshold, strel('disk',n_pixels_to_grow,0));
# imagine(haadf.data, mask, adf.data);

import numpy as np
from skimage.morphology import disk
from scipy.ndimage import binary_dilation
import matplotlib.pyplot as plt

threshold = 28000
n_pixels_to_grow = 1
reproj = tomobase.processes.project(rec_masked, new_sino.angles).data
# reproj: ndarray from your forward projection
mask = reproj > threshold               # boolean mask
selem = disk(n_pixels_to_grow)          # disk structuring element
mask_grown = np.empty_like(mask, dtype=bool)
for i in range(mask.shape[0]):
    mask_grown[i] = binary_dilation(mask[i], structure=selem)

# If you need same dtype as images:
mask_grown_uint = (mask_grown.astype(np.uint8))

stackview.curtain(reproj, mask_grown_uint)

In [ ]:
#stackview.slice(sino.data)
#normalize sino.data
#sino.data = (sino.data - sino.data.min()) / (sino.data.max() - sino.data.min())
print(np.mean(new_sino.data))


In [ ]:
sino2 = tomobase.processes.align_tilt_axis_rotation(new_sino, inplace=False)
stackview.slice(new_sino.data)